In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pymc3 as pm
import math
from numpy import linalg
from datetime import datetime, timedelta
from scipy.interpolate import CubicSpline
from scipy.interpolate import pchip
import matplotlib.pyplot as plt

In [2]:
#Load temperature data
trap_test = 'sample'
trap_station = 'sample'
tempdata=pd.read_csv("data/temp_2016-2019.csv",low_memory=False)
tempdataframe=pd.DataFrame(tempdata)
tempdataframe.set_index('TRAPID')
templist1 = tempdataframe.loc[tempdataframe['TRAPID']==trap_station]
templist1 =np.array(templist1)
templist1 = np.delete(templist1,[0,1,2,3,4,5,6,7,8,9],None)
templist1 = templist1.astype(float)

tempdata2=pd.read_csv("data/temp_2020-2022.csv",low_memory=False)
tempdataframe2=pd.DataFrame(tempdata2)
tempdataframe2.set_index('TRAPID')
templist2=tempdataframe2.loc[tempdataframe['TRAPID']==trap_station]
templist2=np.array(templist2)
templist2=np.delete(templist2,[0,1,2,3,4,5,6,7,8,9],None)
templist2=templist2.astype(float)

templist=[]
for i in range(0,len(templist1)):
    templist.append(templist1[i])
for i in range(0,len(templist2)):
    templist.append(templist2[i])
    
for i in range(0,len(templist)):
    if math.isnan(templist[i])==True:
        templist[i]=templist[i-1]
        
templist_ave=[]
for i in range(1,int(len(templist)/8)):
    templist_ave.append(np.mean(templist[(i-1)*8:i*8]))
    
templist_lw=[]
for i in range(0,14):
    templist_lw.append(templist_ave[i])
for i in range(14,len(templist_ave)):
    templist_lw.append(np.mean(templist_ave[i-14:i]))
    
#Thermal dynamic functions for Culex quinquefaciatus 
#Development rate for larvae (C)
def larvad(T):
    c = 9.14980475e-05
    T0 = 8.39127742e+00
    Tm = 4.02811247e+01
    return np.where((T < T0) | (T > Tm), 0, np.maximum(0, c * T * (T - T0) * abs(T - Tm)**0.5))

#Development rate for pupae (C)
def pupad(T):
    c = 2.81315473e-04
    T0 = 5.68289812e+00
    Tm = 3.85667034e+01
    return np.where((T < T0) | (T > Tm), 0, np.maximum(0, c * T * (T - T0) * abs(T - Tm)**0.5))

def ovirate(T):
    c = 0.23828148
    T0 = 14.689913
    Tm = 40
    return np.where((T < T0) | (T > Tm), 0, np.maximum(0, c * (T - T0) * abs(T - Tm)))

#probability of survival from egg to adult (C)
def pEA(T):
    c = -3.47067577e-03
    T0 = 1.00038210e+01
    Tm = 3.94412177e+01
    return np.where((T < T0) | (T > Tm), 0, np.maximum(0, c * (T - T0) * (T - Tm)))

#death rate of adult (C)
def adultmu(T):
    c = -1.21163521
    T0 = 9
    Tm = 33.45979268
    return np.where((T < T0) | (T > Tm), 1, np.minimum(1, 1/(c * (T - Tm) * abs(T - T0)**0.5)))

#fahrenheit to celcius
def tempfunc(t):
    return (t-32)*5/9

#Define the time-dependent parameters for the three-stage model
larvad_list = []
pupad_list = []
adultmu_list = []
ovirate_list = []
pEA_list = []
for i in range(0,len(templist_lw)):
    larvad_list.append(larvad(tempfunc(templist_lw[i])))
    pupad_list.append(pupad(tempfunc(templist_lw[i])))
    adultmu_list.append(adultmu(tempfunc(templist_lw[i])))
    ovirate_list.append(ovirate(tempfunc(templist_lw[i])))
    pEA_list.append(pEA(tempfunc(templist_lw[i])))
    
xs = np.linspace(0,len(templist_lw)-1,len(templist_lw))
cs_pos_larvad = pchip(xs,larvad_list)
cs_pos_pupad = pchip(xs,pupad_list)
cs_pos_adultmu = pchip(xs,adultmu_list)
cs_pos_ovirate = pchip(xs,ovirate_list)
cs_pos_pEA = pchip(xs,pEA_list)

#Inverse Method
#Use this function to check and see if Acorr can be selected
def majorsign(t):
    return cs_pos_adultmu(t)*cs_pos_pupad(t)*cs_pos_pupad(t)+cs_pos_pupad(t)*cs_pos_adultmu(t,1)-cs_pos_adultmu(t)*cs_pos_pupad(t,1)

#Load mosquito trap data (we provide the aggregated Aedes aegypti data as an example)
mosdata=pd.read_csv("data/2019_2023_mosq_agg_aedes.csv",low_memory=False)
mosdata=pd.DataFrame(mosdata)
datalist = mosdata['Ae_aegypti'][0:36]

timelist = []
for i in range(1,13):
    delta = datetime(2019,i,15)-datetime(2016,8,1)
    timelist.append(float(delta.days))
for i in range(1,13):
    delta = datetime(2020,i,15)-datetime(2016,8,1)
    timelist.append(float(delta.days))
for i in range(1,13):
    delta = datetime(2021,i,15)-datetime(2016,8,1)
    timelist.append(float(delta.days))
    
cs_pos = pchip(timelist,datalist)

xs_obtime = np.linspace(min(timelist),max(timelist),int(max(timelist)-min(timelist)+1))

#Use the Inverse Method to generate time-dependent carrying capacity
def AAsign(t):
    return cs_pos(t,1)+cs_pos_adultmu(t)*cs_pos(t)

A0_list1 = []
for i in range(0,len(xs_obtime)):
    A0_list1.append(AAsign(xs_obtime[i])/cs_pos_adultmu(xs_obtime[i]))
A01 = np.min(A0_list1)
Acorr1 = 2*np.ceil(-A01)

def PPcorr(t):
    return 2*(cs_pos(t,1)+cs_pos_adultmu(t)*(cs_pos(t)+Acorr1))/cs_pos_pupad(t)

PPlist = []
for i in xs_obtime:
    PPlist.append(PPcorr(i))
cs_pos_PP = pchip(xs_obtime,PPlist)

def PPsign(t):
    return cs_pos_PP(t,1)+cs_pos_pupad(t)*cs_pos_PP(t)

A0_list2 = []
for i in range(0,len(xs_obtime)):
    A0_list2.append(PPsign(xs_obtime[i])*cs_pos_pupad(xs_obtime[i])/cs_pos_larvad(xs_obtime[i])/cs_pos_adultmu(xs_obtime[i]))
A02 = np.min(A0_list2)

Acorr2 = np.max(Acorr1, A02)

def PPcorr2(t):
    return 2*(cs_pos(t,1)+cs_pos_adultmu(t)*(cs_pos(t)+Acorr2))/cs_pos_pupad(t)

PPlist2 = []
for i in xs_obtime:
    PPlist2.append(PPcorr2(i))
cs_pos_PP2 = pchip(xs_obtime,PPlist2)

def PPsign2(t):
    return cs_pos_PP2(t,1)+cs_pos_pupad(t)*cs_pos_PP2(t)

#Check to see if A03>0:
A0_list3 = []
for i in range(0,len(xs_obtime)):
    A0_list3.append(PPsign2(xs_obtime[i]))
A03 = np.min(A0_list3)
#If A03>0, replace Acorr2 by np.max(A03,Acorr2).

def LLcorr(t):
    return PPsign2(t)/cs_pos_larvad(t)
LLlist = []
for i in xs:
    LLlist.append(LLcorr(i))
cs_pos_LL = pchip(xs,LLlist)

def capacityK_inv(t):
    return 1/(cs_pos_LL(t)+cs_pos_PP2(t))-(cs_pos_LL(t,1)+cs_pos_PP2(t,1)+cs_pos_pupad(t)*cs_pos_PP2(t))/cs_pos_pEA(t)/cs_pos_ovirate(t)/(cs_pos(t)+Acorr2)/(cs_pos_LL(t)+cs_pos_PP2(t))